# 13.02 OpenAI / Azure OpenAI — 공급자 카탈로그

**`langchain-openai`** 한 패키지로 OpenAI 와 Azure OpenAI 양쪽을 다룬다. 가장 폭넓은 modality 커버리지 — chat · embedding · moderation · DALL-E.

## 학습 목표

- 한 패키지에서 끌어오는 클래스 매핑(ChatOpenAI / OpenAIEmbeddings / AzureChatOpenAI / AzureOpenAIEmbeddings) 파악
- OpenAI 1순위 차별 기능: **Responses API** (`use_responses_api=True`) 와 reasoning effort
- 모델 ID 패밀리: GPT-5 시리즈, o-시리즈(reasoning), embedding-3 시리즈
- 가격·한도·리전 빠른 참조

## 13.02.1 환경 설정

필요 패키지: `langchain-openai`. `.env` 에 `OPENAI_API_KEY`. Azure 사용 시 `AZURE_OPENAI_API_KEY` + `AZURE_OPENAI_ENDPOINT` + 배포명 필요.

```bash
uv pip install langchain-openai
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.environ["OPENAI_API_KEY"], "OPENAI_API_KEY 필요"

## 13.02.2 공급자 제품군 전체 지도

| 카테고리 | 심볼 | 카테고리 노트북 |
|---------|------|----------------|
| Chat | `ChatOpenAI` | [`01_chat_models/01_openai.ipynb`](../01_chat_models/01_openai.ipynb) |
| Chat (Azure) | `AzureChatOpenAI` | [`01_chat_models/01_openai.ipynb`](../01_chat_models/01_openai.ipynb) |
| Embeddings | `OpenAIEmbeddings` | [`02_embeddings/01_openai.ipynb`](../02_embeddings/01_openai.ipynb) |
| Embeddings (Azure) | `AzureOpenAIEmbeddings` | [`02_embeddings/01_openai.ipynb`](../02_embeddings/01_openai.ipynb) |
| Moderation middleware | `OpenAIModerationMiddleware` | [`11_provider_middleware/07_openai_moderation.ipynb`](../11_provider_middleware/07_openai_moderation.ipynb) |
| Image (DALL-E) | `DallEAPIWrapper` (`langchain-community`) | — |
| Speech (Whisper / TTS) | OpenAI SDK 직접 | — |

OpenAI 는 자체 vector store(`File search`/`Vector stores`) 도 보유하지만, LangChain 측에서는 일반 vector store 어댑터 대신 **Responses API + 내장 도구**로 노출한다 (다음 셀).

## 13.02.3 기본 사용 — chat

최소 한 줄. `init_chat_model("openai:...")` 또는 `ChatOpenAI(...)` 직접.

In [ ]:
from langchain_openai import ChatOpenAI

# 2026-04 시점 GA 라인업: gpt-4.1 패밀리 (full / mini / nano)
model = ChatOpenAI(model="gpt-4.1-mini", max_tokens=256)
print(model.invoke("한국어로 한 문장만: 너는 누구냐?").content)

## 13.02.4 공급자 특화 기능 — Responses API

OpenAI 의 1순위 차별 기능은 **Responses API** 다. Chat Completions 의 후속 표준으로, 다음을 한 엔드포인트에서 처리한다.

- **서버 측 대화 상태 보존** — `previous_response_id` 로 이어서 (캐싱 효과)
- **내장 도구**: `web_search`, `file_search`, `code_interpreter`, `image_generation` (벤더 호스팅)
- **Reasoning effort** — o-시리즈에 `reasoning_effort="low"|"medium"|"high"` 로 깊이 조절

LangChain 에서는 `ChatOpenAI(use_responses_api=True)` 한 줄로 활성화. 응답에 `response_metadata['id']` 가 잡히면 다음 호출에서 `use_previous_response_id=True` 로 이어 붙인다.

In [ ]:
from langchain_openai import ChatOpenAI

responses_model = ChatOpenAI(
    model="gpt-4.1-mini",
    use_responses_api=True,            # Responses API 활성화
    use_previous_response_id=True,     # 다음 호출 시 이전 응답 ID 자동 첨부 → 서버 캐시
    max_tokens=128,
)

print(responses_model.invoke("간단 인사").content)

## 13.02.5 가격 · 한도 · 리전

(2026 년 4월 기준 콘솔 표시가의 대략적 위치 — 정확한 단가는 항상 OpenAI 콘솔에서 재확인)

| 모델 | 컨텍스트 | 입력 | 출력 | 비고 |
|------|---------|-----|------|------|
| `gpt-4.1` | 1M | 중상 | 높음 | 최상급 일반 추론 + 긴 컨텍스트 |
| `gpt-4.1-mini` | 1M | 저렴 | 저렴 | 에이전트 기본값 |
| `gpt-4.1-nano` | 1M | 매우 저렴 | 매우 저렴 | 분류·라우팅·요약 |
| `o4-mini` | 200k | 중간 | 중간 | 강한 reasoning, `reasoning_effort` 지원 |
| `text-embedding-3-large` | 8191 | $0.13 / 1M | — | 3072d (축소 가능) |
| `text-embedding-3-small` | 8191 | $0.02 / 1M | — | 1536d |

리전:

- Direct API: 글로벌 단일 엔드포인트
- Azure OpenAI: 동·서·중부 US, EU, JP/KR 등 — **모델 가용성이 리전마다 다르다**, 배포(deployment)를 만들어야 사용

한도: Tier 별 RPM/TPM. Tier 1 신규 계정은 분당 ~500 RPM. Tier 5 까지 자동 승격.

## 핵심 정리

- **chat 1순위**: 일반 에이전트 = `gpt-4.1-mini`, reasoning 필요 = `o4-mini` + `reasoning_effort`, 분류 = `gpt-4.1-nano`
- **embedding 1순위**: 한국어/다국어 = `text-embedding-3-large`, 비용 우선 = `-small` (`dimensions=512` 로 추가 축소)
- **Responses API** 는 멀티턴/내장 도구 워크로드에서 점차 기본 경로로 자리잡는 중 — `use_responses_api=True` 채택
- Azure 경유는 `deployment` 명을 모델 ID 대신 사용 — 별도 endpoint/키 환경변수

## 다음

- [`01_chat_models/01_openai.ipynb`](../01_chat_models/01_openai.ipynb) — ChatOpenAI 깊은 사용 (Responses · structured output)
- [`02_embeddings/01_openai.ipynb`](../02_embeddings/01_openai.ipynb) — text-embedding-3 series · `dimensions` 축소
- [`11_provider_middleware/07_openai_moderation.ipynb`](../11_provider_middleware/07_openai_moderation.ipynb) — Moderation API 미들웨어

## 참고

- `langchain-openai` PyPI: https://pypi.org/project/langchain-openai/
- 공식 docs: https://docs.langchain.com/oss/python/integrations/providers/openai
- OpenAI Responses API: https://platform.openai.com/docs/api-reference/responses
- Azure OpenAI 리전·모델 매트릭스: https://learn.microsoft.com/azure/ai-services/openai/concepts/models